<a href="https://colab.research.google.com/github/nazndev/visibility-gated-clip-reid/blob/main/notebooks/colab_first_iteration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import subprocess
from pathlib import Path

root = Path("/content/lvlm-person-reid")
if not (root / "src" / "reidvlm").is_dir():
    subprocess.check_call([
        "git", "clone",
        "https://github.com/nazndev/visibility-gated-clip-reid.git",
        str(root),
    ])
else:
    subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only"])
os.environ["REID_ROOT"] = str(root)
os.chdir(root)
print(root)


/content/lvlm-person-reid


In [2]:
import os

print("nvidia device", os.path.exists("/dev/nvidia0"))
if not os.path.exists("/dev/nvidia0"):
    raise RuntimeError("Select a GPU runtime, then run this cell again.")


nvidia device True


In [ ]:
!cd "$REID_ROOT" && pip install -q -r requirements-colab.txt && pip install -q -e .
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime and run this cell again.")


In [ ]:
import os
import zipfile
from pathlib import Path

import gdown

root = Path(os.environ["REID_ROOT"])
data = root / "data" / "Market-1501-v15.09.15"
if not (data / "bounding_box_train").is_dir():
    archive = root / "data" / "market1501.zip"
    archive.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=str(archive))
    if not archive.is_file():
        raise FileNotFoundError("The Market-1501 download did not produce market1501.zip.")
    with zipfile.ZipFile(archive) as package:
        package.extractall(root / "data")
print(data)
print("train", (data / "bounding_box_train").is_dir())


In [ ]:
import hashlib
import os
from pathlib import Path

import gdown

root = Path(os.environ["REID_ROOT"])
checkpoint = root / "checkpoints" / "Market1501_clipreid_ViT-B-16_60.pth"
expected = "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc"
if not checkpoint.is_file():
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id="1GnyAVeNOg3Yug1KBBWMKKbT2x43O5Ch7", output=str(checkpoint))
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint.name)
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
print(digest)
if digest != expected:
    raise RuntimeError("Checkpoint hash does not match the official CLIP-ReID file.")


In [ ]:
!cd "$REID_ROOT" && python -m reidvlm verify-data --config configs/market1501.yaml
!cd "$REID_ROOT" && python -m reidvlm verify-checkpoint --config configs/market1501.yaml


In [ ]:
print("CLIP-ReID ranking of all Market-1501 queries.")
!cd "$REID_ROOT" && python -m reidvlm evaluate --config configs/market1501.yaml --visual-only


In [ ]:
print("Token pruning on one query. Half the patches are dropped after layer 6.")
print("This is not a Market-1501 score.")
!cd "$REID_ROOT" && python scripts/07_token_pruning.py


In [ ]:
print("CLIP-ReID ranking of all Market-1501 queries with token pruning.")
print("Half the patch tokens are dropped after layer 6. Qwen is not called.")
!cd "$REID_ROOT" && python -m reidvlm evaluate --config configs/market1501.yaml --visual-only --keep-ratio 0.5 --prune-layer 6


In [ ]:
print("Qwen judges one query. This is not a Market-1501 score.")
!cd "$REID_ROOT" && python scripts/06_qwen_one_query.py


In [ ]:
import os
import subprocess
from pathlib import Path

print("Full Qwen judgment of all 3,368 queries. This takes many days.")
print("Token pruning is already finished in the cells above.")
print("Leave RUN_FULL_QWEN = False unless the GPU can stay on for days.")
RUN_FULL_QWEN = False
if not RUN_FULL_QWEN:
    print("Skipped.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    path = Path("/content/lvlm-person-reid/configs/market1501.yaml")
    text = path.read_text()
    local_cache = "cache_dir: caches/judgments"
    drive_cache = "cache_dir: /content/drive/MyDrive/visibility-gated-reid/judgments"
    if local_cache in text:
        path.write_text(text.replace(local_cache, drive_cache))
    elif drive_cache not in text:
        raise SystemExit("cache_dir line was not found")
    saved = Path("/content/drive/MyDrive/visibility-gated-reid/judgments")
    print(len(list(saved.glob("*.json"))) if saved.is_dir() else 0)
    os.chdir("/content/lvlm-person-reid")
    os.environ["HF_HOME"] = "/content/lvlm-person-reid/caches/huggingface"
    subprocess.check_call(["python", "-m", "reidvlm", "evaluate", "--config", "configs/market1501.yaml"])
